In [4]:
# ============================================================
# SALARY SURVEY 2021
# DATA CLEANING AND WRANGLING
# ============================================================

import re
import numpy as np
import pandas as pd

In [7]:
# ============================================================
# 1. LOAD DATASET
# ============================================================

INPUT_FILE = "raw_data/Salary Survey 2021.csv"
OUTPUT_FILE = "salary_survey_cleaned.csv"

df = pd.read_csv('Ask A Manager Salary Survey 2021 (Responses).csv')

# Keep an untouched copy of the original dataset
df_raw = df.copy()

print("=" * 60)
print("INITIAL DATASET")
print("=" * 60)

print("Shape:", df.shape)
print()

print("Data Information:")
df.info()
print()

print("Missing Values:")
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

print(missing_summary)
print()


INITIAL DATASET
Shape: (28225, 18)

Data Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28225 entries, 0 to 28224
Data columns (total 18 columns):
 #   Column                                                                                                                                                                                                                                Non-Null Count  Dtype  
---  ------                                                                                                                                                                                                                                --------------  -----  
 0   Timestamp                                                                                                                                                                                                                             28225 non-null  object 
 1   How old are you?                                       

In [8]:


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def print_shape(option="both"):
    """
    Print the current number of rows and/or columns
    compared with the original dataset.
    """

    raw_rows, raw_cols = df_raw.shape
    current_rows, current_cols = df.shape

    option = option.lower()

    if option in ["row", "both"]:
        print(f"Total Rows: {current_rows}/{raw_rows}")

    if option in ["col", "both"]:
        print(f"Total Columns: {current_cols}/{raw_cols}")


def print_unique_summary(df_name, column_name, max_display=15):
    """
    Print the number of unique values and display
    the unique values when the list is reasonably small.
    """

    unique_values = df_name[column_name].unique()
    unique_count = df_name[column_name].nunique()

    print(f"\nColumn: {column_name}")
    print(f"Total Unique Count: {unique_count}")

    if unique_count <= max_display:
        print("Unique Values:", unique_values)
    else:
        print(
            f"First {max_display} Unique Values:",
            unique_values[:max_display]
        )


def print_null_sum(df_name, column_name):
    """
    Print the number of NULL values in a column.
    """

    print(
        f"{column_name} NULL Values:",
        df_name[column_name].isnull().sum()
    )


def map_category(value, mapping):
    """
    Map a text value into a predefined category
    using regular-expression patterns.
    """

    if pd.isna(value):
        return value

    value = str(value).strip()

    for category, pattern in mapping.items():

        if pd.Series([value]).str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        ).iloc[0]:

            return category

    return "Other"

In [9]:
# ============================================================
# 3. REMOVE UNNECESSARY COLUMNS
# ============================================================

columns_to_drop = [
    "Timestamp",
    "If your job title needs additional context, please clarify here:",
    "Additional monetary compensation",
    'If "Other," please indicate the currency here:',
    "If your income needs additional context, please provide it here:",
    "US State",
    "City",
    "Overall years of professional experience",
    "Race"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

print("=" * 60)
print("AFTER COLUMN REMOVAL")
print("=" * 60)

print_shape()
print()


AFTER COLUMN REMOVAL
Total Rows: 28225/28225
Total Columns: 15/18



In [10]:

# ============================================================
# 4. RENAME COLUMNS
# ============================================================

column_rename = {
    "Age Range": "age_range",
    "Industry": "industry",
    "Job Title": "job_title",
    "Annual Salary": "annual_salary",
    "Currency": "currency",
    "Country": "country",
    "Years of professional experience in your current field": "current_years_experience",
    "Highest Level of Education Completed": "education",
    "Gender": "gender"
}

df = df.rename(columns=column_rename)

print("=" * 60)
print("RENAMED COLUMNS")
print("=" * 60)

print(df.columns.tolist())
print()


RENAMED COLUMNS
['How old are you?', 'What industry do you work in?', 'Job title', "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)", 'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.', 'Please indicate the currency', 'If "Other," please indicate the currency here: ', 'What country do you work in?', "If you're in the U.S., what state do you work in?", 'What city do you work in?', 'How many years of professional work experience do you have overall?', 'How many years of professional work experience do you have in your field?', 'What is your highest level of education completed?', 'What is your gender?', 'What is your race? (Choose all that apply.)']



In [17]:
# ============================================================
# 5. INDUSTRY CLEANING
# ============================================================

print("=" * 60)
print("INDUSTRY CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 5.1 RENAME INDUSTRY COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "What industry do you work in?": "industry"
})


# ------------------------------------------------------------
# 5.2 FUNCTION TO DISPLAY UNIQUE VALUES
# ------------------------------------------------------------

print("\nOriginal Industry Values:")

print("Column:", "industry")
print("Total Unique Count:", df["industry"].nunique())

print("First 15 Unique Values:")
print(df["industry"].dropna().unique()[:15])


# ------------------------------------------------------------
# 5.3 INDUSTRY CATEGORY MAPPING
# ------------------------------------------------------------

industry_mapping = {

    "Contact Center Services":
        r"call center|contact center|customer service|customer support",

    "Education":
        r"education|school|university|college|academic|higher education|primary|secondary",

    "Health":
        r"health|health care|healthcare|hospital|medical|medicine|nursing|clinical",

    "Biotech":
        r"biotech|biotechnology|pharmaceutical|pharma",

    "Legal":
        r"legal|law firm|lawyer|attorney|\blaw\b",

    "Government":
        r"government|public sector|federal|state government|municipal|public administration",

    "Nonprofit":
        r"nonprofit|non-profit|charity|ngo",

    "Marketing":
        r"marketing|advertising|public relations|\bpr agency\b",

    "Retail/Sales":
        r"retail|sales|e-commerce|ecommerce|merchandising",

    "Engineering/Manufacturing":
        r"engineering|manufacturing|mechanical|electrical|industrial|production|automotive|aerospace",

    "Media/Entertainment":
        r"media|entertainment|film|television|broadcast|music|gaming|digital media|publishing",

    "Hospitality":
        r"hospitality|hotel|restaurant|food service|travel|tourism",

    "Arts/Design":
        r"\bart\b|arts|design|fashion|creative",

    "Real Estate":
        r"real estate|property|housing",

    "Transportation":
        r"transportation|logistics|shipping|aviation|airline",

    "Agriculture":
        r"agriculture|farming|agricultural|food production",

    "Energy":
        r"energy|oil|gas|utilities|renewable",

    "Insurance":
        r"insurance",

    "Veterinary":
        r"veterinary|veterinarian|animal health",

    "Business/Consulting":
        r"consulting|consultant|business services|professional services|business or consulting",

    "Information Technology":
        r"information technology|information systems|software|technology|tech|computer|\bit\b|computing",

    "Finance":
        r"finance|financial|banking|bank|investment|accounting|fintech"
}


# ------------------------------------------------------------
# 5.4 FUNCTION TO MAP INDUSTRY INTO CATEGORIES
# ------------------------------------------------------------

def map_category(value, mapping):

    # Handle missing values
    if pd.isna(value):
        return "Not Specified"

    # Convert value to lowercase text
    value = str(value).strip().lower()

    # Check each category
    for category, pattern in mapping.items():

        if re.search(pattern, value, flags=re.IGNORECASE):
            return category

    # If no category matches
    return "Other"


# ------------------------------------------------------------
# 5.5 APPLY INDUSTRY CATEGORIES
# ------------------------------------------------------------

print("\nMapping industries into categories...")

df["industry"] = df["industry"].apply(
    lambda x: map_category(x, industry_mapping)
)

print("Industry mapping completed.")


# ------------------------------------------------------------
# 5.6 HANDLE MISSING VALUES
# ------------------------------------------------------------

df["industry"] = df["industry"].fillna("Not Specified")


# ------------------------------------------------------------
# 5.7 DISPLAY CLEANED INDUSTRY RESULTS
# ------------------------------------------------------------

print("\nIndustry categories after cleaning:")

print(
    df["industry"]
    .value_counts()
)


# ------------------------------------------------------------
# 5.8 NUMBER OF INDUSTRY CATEGORIES
# ------------------------------------------------------------

print("\nNumber of Industry Categories:")

print(
    df["industry"].nunique()
)


# ------------------------------------------------------------
# 5.9 FINAL CHECK
# ------------------------------------------------------------

print("\nIndustry Cleaning Complete.")

print("=" * 60)

INDUSTRY CLEANING

Original Industry Values:
Column: industry
Total Unique Count: 1225
First 15 Unique Values:
['Education (Higher Education)' 'Computing or Tech'
 'Accounting, Banking & Finance' 'Nonprofits' 'Publishing'
 'Education (Primary/Secondary)' 'Law' 'Health care'
 'Utilities & Telecommunications' 'Business or Consulting' 'Art & Design'
 'Government and Public Administration' 'Public Library'
 'Engineering or Manufacturing' 'Media & Digital']

Mapping industries into categories...
Industry mapping completed.

Industry categories after cleaning:
industry
Information Technology       4778
Education                    3409
Nonprofit                    2432
Health                       2275
Government                   1937
Other                        1868
Finance                      1833
Engineering/Manufacturing    1826
Legal                        1150
Marketing                    1143
Media/Entertainment          1121
Business/Consulting           904
Retail/Sales          

In [19]:
# ============================================================
# 6. JOB TITLE CLEANING
# ============================================================

print("=" * 60)
print("JOB TITLE CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 6.1 RENAME JOB TITLE COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "Job title": "job_title"
})


# ------------------------------------------------------------
# 6.2 CHECK ORIGINAL JOB TITLE VALUES
# ------------------------------------------------------------

print("\nOriginal Job Title Values:")

print("Column:", "job_title")
print("Total Unique Count:", df["job_title"].nunique())

print("First 15 Unique Values:")
print(df["job_title"].dropna().unique()[:15])


# ------------------------------------------------------------
# 6.3 JOB TITLE CATEGORY MAPPING
# ------------------------------------------------------------

job_title_mapping = {

    # --------------------------------------------------------
    # TECHNOLOGY
    # --------------------------------------------------------

    "Software Engineer": (
        r"software engineer|software developer|web developer|"
        r"frontend developer|front-end developer|backend developer|"
        r"back-end developer|full stack developer|full-stack developer"
    ),

    "Data Scientist/Analyst": (
        r"data scientist|data analyst|data engineer|"
        r"business intelligence|bi analyst"
    ),

    "IT/Systems": (
        r"information technology|it specialist|it support|"
        r"systems administrator|system administrator|"
        r"systems analyst|network administrator|network engineer"
    ),

    "DevOps/Cloud": (
        r"devops|dev ops|cloud engineer|cloud architect|"
        r"site reliability engineer|\bsre\b"
    ),

    "QA/Test Engineer": (
        r"quality assurance|qa engineer|qa analyst|"
        r"test engineer|software tester"
    ),


    # --------------------------------------------------------
    # PRODUCT / PROJECT
    # --------------------------------------------------------

    "Product Manager":
        r"product manager",

    "Project Manager":
        r"project manager|program manager",

    "UX/UI Designer": (
        r"ux designer|ui designer|ux/ui|user experience|"
        r"user interface"
    ),


    # --------------------------------------------------------
    # SENIOR LEADERSHIP
    # --------------------------------------------------------

    "VP/Senior Leadership": (
        r"\bvp\b|vice president|chief executive|"
        r"chief technology|chief financial|chief operating|"
        r"chief information|chief marketing|c-suite"
    ),

    "Director":
        r"\bdirector\b",


    # --------------------------------------------------------
    # SPECIFIC MANAGEMENT
    # --------------------------------------------------------

    "Sales Manager":
        r"sales manager",

    "Marketing Manager":
        r"marketing manager",

    "HR Generalist/Manager": (
        r"human resources|hr generalist|hr manager|"
        r"human resource manager"
    ),


    # --------------------------------------------------------
    # EXECUTIVE / OFFICE
    # --------------------------------------------------------

    "Executive Assistant":
        r"executive assistant",

    "Administrative Assistant": (
        r"administrative assistant|admin assistant"
    ),

    "Office Manager":
        r"office manager",

    "Receptionist":
        r"receptionist|front desk",

    "Coordinator":
        r"\bcoordinator\b",


    # --------------------------------------------------------
    # FINANCE
    # --------------------------------------------------------

    "Accountant":
        r"accountant",

    "Financial Analyst":
        r"financial analyst",

    "Bookkeeper":
        r"bookkeeper|book keeping|bookkeeping",

    "Auditor":
        r"auditor",

    "Controller":
        r"\bcontroller\b",


    # --------------------------------------------------------
    # SALES
    # --------------------------------------------------------

    "Sales Representative": (
        r"sales representative|sales rep|account executive"
    ),


    # --------------------------------------------------------
    # MARKETING
    # --------------------------------------------------------

    "Content/Copywriter": (
        r"copywriter|content writer|content specialist"
    ),

    "Social Media":
        r"social media",


    # --------------------------------------------------------
    # EDUCATION
    # --------------------------------------------------------

    "Teacher":
        r"\bteacher\b|educator",

    "Professor/Faculty": (
        r"professor|faculty|lecturer|instructor"
    ),

    "Librarian":
        r"librarian",


    # --------------------------------------------------------
    # CUSTOMER SERVICE
    # --------------------------------------------------------

    "Customer Service": (
        r"customer service|customer support|client service"
    ),


    # --------------------------------------------------------
    # CREATIVE
    # --------------------------------------------------------

    "Graphic Designer": (
        r"graphic designer|visual designer"
    ),

    "Writer/Editor": (
        r"\bwriter\b|editor|technical writer"
    ),


    # --------------------------------------------------------
    # PROFESSIONAL
    # --------------------------------------------------------

    "Consultant":
        r"consultant|consulting",

    "Social Worker":
        r"social worker",


    # --------------------------------------------------------
    # GENERAL LEADERSHIP
    # --------------------------------------------------------

    "Team Lead":
        r"team lead|team leader",

    "Supervisor":
        r"\bsupervisor\b",

    "Manager (General)":
        r"\bmanager\b",


    # --------------------------------------------------------
    # GENERAL ANALYST
    # --------------------------------------------------------

    "Analyst (General)":
        r"\banalyst\b",


    # --------------------------------------------------------
    # NON-SOFTWARE ENGINEERING
    # --------------------------------------------------------

    "Engineer (Non-software)":
        r"engineer|engineering"
}


# ------------------------------------------------------------
# 6.4 APPLY JOB TITLE CATEGORIES
# ------------------------------------------------------------

print("\nMapping job titles into categories...")

df["job_title"] = df["job_title"].apply(
    lambda x: map_category(x, job_title_mapping)
)

print("Job title mapping completed.")


# ------------------------------------------------------------
# 6.5 HANDLE MISSING VALUES
# ------------------------------------------------------------

df["job_title"] = df["job_title"].fillna("Not Specified")


# ------------------------------------------------------------
# 6.6 DISPLAY CLEANED JOB TITLE RESULTS
# ------------------------------------------------------------

print("\nJob Title categories after cleaning:")

print(
    df["job_title"].value_counts()
)


# ------------------------------------------------------------
# 6.7 NUMBER OF JOB TITLE CATEGORIES
# ------------------------------------------------------------

print("\nNumber of Job Title Categories:")

print(
    df["job_title"].nunique()
)


# ------------------------------------------------------------
# 6.8 FINAL CHECK
# ------------------------------------------------------------

print("\nJob Title Cleaning Complete.")

print("=" * 60)

JOB TITLE CLEANING

Original Job Title Values:
Column: job_title
Total Unique Count: 14434
First 15 Unique Values:
['Research and Instruction Librarian'
 'Change & Internal Communications Manager' 'Marketing Specialist'
 'Program Manager' 'Accounting Manager' 'Scholarly Publishing Librarian'
 'Publishing Assistant' 'Librarian' 'Systems Analyst' 'Senior Accountant'
 'Office Manager'
 'Deputy Title IX Coordinator/ Assistant Director Office of Equity and Diversity'
 'Manager of Information Services' 'Legal Aid Staff Attorney'
 'Patient care coordinator']

Mapping job titles into categories...
Job title mapping completed.

Job Title categories after cleaning:
job_title
Other                       9625
Manager (General)           3382
Director                    2598
Analyst (General)           1241
Software Engineer           1233
Engineer (Non-software)     1158
Coordinator                 1049
Project Manager              886
Writer/Editor                627
Consultant                   

In [20]:
# ============================================================
# 7. ANNUAL SALARY CLEANING
# ============================================================

print("=" * 60)
print("ANNUAL SALARY CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 7.1 RENAME SALARY COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)": "annual_salary"
})


# ------------------------------------------------------------
# 7.2 CHECK ORIGINAL SALARY DATA TYPE
# ------------------------------------------------------------

print("Original salary data type:")

print(df["annual_salary"].dtype)


# ------------------------------------------------------------
# 7.3 CONVERT SALARY TO NUMERIC
# ------------------------------------------------------------

df["annual_salary"] = pd.to_numeric(
    df["annual_salary"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip(),
    errors="coerce"
)


print("\nSalary data type after conversion:")

print(df["annual_salary"].dtype)


print("\nSalary conversion NULL values:")

print(df["annual_salary"].isnull().sum())


print("\nSalary statistics:")

print(df["annual_salary"].describe())

print()


ANNUAL SALARY CLEANING
Original salary data type:
object

Salary data type after conversion:
Int64

Salary conversion NULL values:
0

Salary statistics:
count            28225.0
mean         371137.9307
std      36133664.256818
min                  0.0
25%              54000.0
50%              75000.0
75%             109450.0
max         6000070000.0
Name: annual_salary, dtype: Float64



In [22]:
# ============================================================
# 8. CURRENCY CLEANING
# ============================================================

print("=" * 60)
print("CURRENCY CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 8.1 RENAME CURRENCY COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "Please indicate the currency": "currency"
})


# ------------------------------------------------------------
# 8.2 STANDARDIZE CURRENCY VALUES
# ------------------------------------------------------------

df["currency"] = (
    df["currency"]
    .astype("string")
    .str.strip()
)


print("Currency counts:")

print(df["currency"].value_counts())


print("\nNumber of 'Other' currency records:")

print(
    (df["currency"].str.lower() == "other").sum()
)


# ------------------------------------------------------------
# 8.3 REMOVE UNKNOWN CURRENCIES
# ------------------------------------------------------------

# Records with "Other" currency are removed because
# their salary cannot be reliably compared with
# standard currency categories without conversion.

df = df[
    df["currency"].str.lower() != "other"
].copy()


print("\nShape after removing 'Other' currency:")

print(df.shape)

print()




CURRENCY CLEANING
Currency counts:
currency
USD        23497
CAD         1677
GBP         1598
EUR          654
AUD/NZD      505
CHF           37
SEK           37
JPY           23
ZAR           17
HKD            4
Name: count, dtype: Int64

Number of 'Other' currency records:
0

Shape after removing 'Other' currency:
(28049, 15)



In [23]:
# ============================================================
# 9. COUNTRY CLEANING
# ============================================================

print("=" * 60)
print("COUNTRY CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 9.1 RENAME COUNTRY COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "What country do you work in?": "country"
})


# ------------------------------------------------------------
# 9.2 STANDARDIZE COUNTRY VALUES
# ------------------------------------------------------------

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)


print("Original unique country count:")

print(df["country"].nunique())


# ------------------------------------------------------------
# 9.3 COUNTRY MAPPING
# ------------------------------------------------------------

country_mapping = {

    "United States":
        r"united states|^usa$|^us$|u\.?s\.?a?\.?$",

    "United Kingdom":
        r"united kingdom|^uk$|england|scotland|wales|britain",

    "Canada":
        r"canada",

    "Australia":
        r"australia",

    "Germany":
        r"germany|deutschland",

    "India":
        r"india",

    "Ireland":
        r"ireland",

    "France":
        r"france",

    "Netherlands":
        r"netherlands|holland",

    "New Zealand":
        r"new zealand"
}


# ------------------------------------------------------------
# 9.4 APPLY COUNTRY MAPPING
# ------------------------------------------------------------

df["country"] = df["country"].apply(
    lambda x: map_category(x, country_mapping)
)


df["country"] = df["country"].fillna(
    "Not Specified"
)


print("\nCountry categories after cleaning:")

print(df["country"].value_counts())

print()

COUNTRY CLEANING
Original unique country count:
309

Country categories after cleaning:
country
United States     23132
Canada             1684
United Kingdom     1579
Other               649
Australia           386
Germany             199
Ireland             128
New Zealand         124
Netherlands          89
France               69
India                 9
Not Specified         1
Name: count, dtype: int64



In [24]:
# ============================================================
# 10. CURRENT YEARS OF EXPERIENCE CLEANING
# ============================================================

print("=" * 60)
print("CURRENT YEARS OF EXPERIENCE")
print("=" * 60)


# ------------------------------------------------------------
# 10.1 RENAME EXPERIENCE COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "How many years of professional work experience do you have overall?":
        "current_years_experience"
})


print("Original experience categories:")

print(
    df["current_years_experience"]
    .dropna()
    .unique()
)


# ------------------------------------------------------------
# 10.2 FUNCTION TO EXTRACT FIRST NUMBER
# ------------------------------------------------------------

def first_number(label):
    """
    Extract the first number from an experience category.
    Used to sort experience groups logically.
    """

    if pd.isna(label):
        return np.inf

    match = re.search(
        r"\d+",
        str(label)
    )

    if match:
        return int(match.group())

    return np.inf


# ------------------------------------------------------------
# 10.3 CREATE EXPERIENCE ORDER
# ------------------------------------------------------------

experience_order = sorted(
    df["current_years_experience"]
    .dropna()
    .unique(),
    key=first_number
)


# ------------------------------------------------------------
# 10.4 APPLY ORDERED CATEGORY
# ------------------------------------------------------------

df["current_years_experience"] = pd.Categorical(
    df["current_years_experience"],
    categories=experience_order,
    ordered=True
)


print("\nExperience categories after ordering:")

print(
    df["current_years_experience"]
    .cat.categories
)

print()

CURRENT YEARS OF EXPERIENCE
Original experience categories:
['5-7 years' '8 - 10 years' '2 - 4 years' '21 - 30 years' '11 - 20 years'
 '1 year or less' '41 years or more' '31 - 40 years']

Experience categories after ordering:
Index(['1 year or less', '2 - 4 years', '5-7 years', '8 - 10 years',
       '11 - 20 years', '21 - 30 years', '31 - 40 years', '41 years or more'],
      dtype='object')



In [25]:
# ============================================================
# 11. EDUCATION CLEANING
# ============================================================

print("=" * 60)
print("EDUCATION CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 11.1 RENAME EDUCATION COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "What is your highest level of education completed?":
        "education"
})


# ------------------------------------------------------------
# 11.2 HANDLE MISSING EDUCATION VALUES
# ------------------------------------------------------------

print("Education NULL values before cleaning:")

print(
    df["education"].isnull().sum()
)


# Missing education information is retained as
# "Not Specified" rather than assigning a guessed value.

df["education"] = (
    df["education"]
    .astype("string")
    .str.strip()
    .fillna("Not Specified")
)


print("\nEducation categories:")

print(
    df["education"].value_counts()
)

print()


EDUCATION CLEANING
Education NULL values before cleaning:
236

Education categories:
education
College degree                        13486
Master's degree                        8858
Some college                           2081
PhD                                    1421
Professional degree (MD, JD, etc.)     1319
High School                             648
Not Specified                           236
Name: count, dtype: Int64



In [26]:
# ============================================================
# 12. GENDER CLEANING
# ============================================================

print("=" * 60)
print("GENDER CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 12.1 RENAME GENDER COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "What is your gender?":
        "gender"
})


# ------------------------------------------------------------
# 12.2 HANDLE MISSING GENDER VALUES
# ------------------------------------------------------------

print("Gender NULL values before cleaning:")

print(
    df["gender"].isnull().sum()
)


# ------------------------------------------------------------
# 12.3 STANDARDIZE GENDER CATEGORY
# ------------------------------------------------------------

df["gender"] = df["gender"].replace(
    {
        "Other or prefer not to answer":
            "Prefer not to answer"
    }
)


df["gender"] = (
    df["gender"]
    .astype("string")
    .str.strip()
    .fillna("Not Specified")
)


print("\nGender categories after cleaning:")

print(
    df["gender"].value_counts()
)

print()


GENDER CLEANING
Gender NULL values before cleaning:
186

Gender categories after cleaning:
gender
Woman                   21319
Man                      5505
Non-binary                745
Prefer not to answer      294
Not Specified             186
Name: count, dtype: Int64



In [27]:
# ============================================================
# 13. AGE RANGE ORDERING
# ============================================================

print("=" * 60)
print("AGE RANGE CLEANING")
print("=" * 60)


# ------------------------------------------------------------
# 13.1 RENAME AGE COLUMN
# ------------------------------------------------------------

df = df.rename(columns={
    "How old are you?":
        "age_range"
})


# ------------------------------------------------------------
# 13.2 STANDARDIZE AGE VALUES
# ------------------------------------------------------------

df["age_range"] = (
    df["age_range"]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# 13.3 DEFINE AGE ORDER
# ------------------------------------------------------------

age_order = [
    "under 18",
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65 or over"
]


# ------------------------------------------------------------
# 13.4 FIND ACTUAL VALUES IN DATASET
# ------------------------------------------------------------

actual_age_values = (
    df["age_range"]
    .dropna()
    .unique()
)


age_lookup = {
    str(value).strip().lower(): value
    for value in actual_age_values
}


ordered_existing_values = [
    age_lookup[age]
    for age in age_order
    if age in age_lookup
]


# ------------------------------------------------------------
# 13.5 APPLY ORDERED AGE CATEGORY
# ------------------------------------------------------------

if ordered_existing_values:

    df["age_range"] = pd.Categorical(
        df["age_range"],
        categories=ordered_existing_values,
        ordered=True
    )


print("Age range values:")

print(
    df["age_range"].value_counts()
)

print()

AGE RANGE CLEANING
Age range values:
age_range
25-34         12613
35-44          9872
45-54          3178
18-24          1279
55-64           994
65 or over       95
under 18         18
Name: count, dtype: int64



In [28]:
# ============================================================
# 14. SALARY OUTLIER DETECTION
# ============================================================

print("=" * 60)
print("SALARY OUTLIER DETECTION")
print("=" * 60)


# ------------------------------------------------------------
# 14.1 SALARY STATISTICS BEFORE OUTLIER REMOVAL
# ------------------------------------------------------------

print("Salary statistics before outlier removal:")

print(
    df["annual_salary"].describe()
)


# ------------------------------------------------------------
# 14.2 DEFINE DOMAIN BOUNDARIES
# ------------------------------------------------------------

# The survey contains extremely large salary values.
# Fixed domain boundaries are used instead of relying
# only on the statistical IQR method.

LOWER_BOUND = 10_000
UPPER_BOUND = 2_000_000


# ------------------------------------------------------------
# 14.3 CREATE OUTLIER FLAG
# ------------------------------------------------------------

df["salary_outlier_flag"] = ~df[
    "annual_salary"
].between(
    LOWER_BOUND,
    UPPER_BOUND
)


# ------------------------------------------------------------
# 14.4 CALCULATE OUTLIER STATISTICS
# ------------------------------------------------------------

outlier_count = (
    df["salary_outlier_flag"]
    .sum()
)


total_records = len(df)


if total_records > 0:

    outlier_percentage = (
        outlier_count /
        total_records *
        100
    )

else:

    outlier_percentage = 0


print("\nSalary outlier lower bound:")

print(LOWER_BOUND)


print("\nSalary outlier upper bound:")

print(UPPER_BOUND)


print("\nNumber of salary outliers:")

print(outlier_count)


print("\nPercentage of salary outliers:")

print(
    round(
        outlier_percentage,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 14.5 DISPLAY EXAMPLES OF OUTLIERS
# ------------------------------------------------------------

print("\nExamples of salary outliers:")

print(
    df.loc[
        df["salary_outlier_flag"],
        [
            "annual_salary",
            "currency",
            "country",
            "job_title"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 14.6 SAVE OUTLIERS SEPARATELY
# ------------------------------------------------------------

df.loc[
    df["salary_outlier_flag"]
].to_csv(
    "salary_outliers.csv",
    index=False
)


print("\nSalary outliers saved as:")

print("salary_outliers.csv")


# ------------------------------------------------------------
# 14.7 CREATE ANALYSIS-READY DATASET
# ------------------------------------------------------------

df_analysis_ready = (
    df[
        ~df["salary_outlier_flag"]
    ]
    .drop(
        columns=["salary_outlier_flag"]
    )
    .copy()
)


print("\nShape after salary outlier removal:")

print(
    df_analysis_ready.shape
)

print()

SALARY OUTLIER DETECTION
Salary statistics before outlier removal:
count            28049.0
mean       314407.575208
std      35832600.797534
min                  0.0
25%              54000.0
50%              75000.0
75%             108055.0
max         6000070000.0
Name: annual_salary, dtype: Float64

Salary outlier lower bound:
10000

Salary outlier upper bound:
2000000

Number of salary outliers:
208

Percentage of salary outliers:
0.74 %

Examples of salary outliers:
      annual_salary currency         country               job_title
97               58      USD   United States        QA/Test Engineer
166              35      EUR           Other                   Other
895              38      USD   United States                 Teacher
968              61      USD   United States           Writer/Editor
1030           4400      GBP  United Kingdom       Manager (General)
1607            130      USD   United States  Data Scientist/Analyst
2124        3000000      USD   United Sta

In [29]:
# ============================================================
# 15. FINAL DATA TYPE CLEANING
# ============================================================

print("=" * 60)
print("FINAL DATA TYPES")
print("=" * 60)


# ------------------------------------------------------------
# 15.1 SELECT CATEGORICAL COLUMNS
# ------------------------------------------------------------

category_columns = [
    "industry",
    "job_title",
    "currency",
    "country",
    "education",
    "gender"
]


# ------------------------------------------------------------
# 15.2 CONVERT COLUMNS TO CATEGORY
# ------------------------------------------------------------

for column in category_columns:

    df_analysis_ready[column] = (
        df_analysis_ready[column]
        .astype("category")
    )


# ------------------------------------------------------------
# 15.3 REAPPLY ORDERED EXPERIENCE CATEGORY
# ------------------------------------------------------------

df_analysis_ready[
    "current_years_experience"
] = pd.Categorical(
    df_analysis_ready[
        "current_years_experience"
    ],
    categories=experience_order,
    ordered=True
)


# ------------------------------------------------------------
# 15.4 DISPLAY FINAL DATA TYPES
# ------------------------------------------------------------

print(
    df_analysis_ready.dtypes
)

print()

FINAL DATA TYPES
age_range                                                                                                                                                                                         category
industry                                                                                                                                                                                          category
job_title                                                                                                                                                                                         category
annual_salary                                                                                                                                                                                        Int64
How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the

In [ ]:
# ============================================================
# 16. FINAL VALIDATION
# ============================================================

print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)


# ------------------------------------------------------------
# 16.1 FINAL SHAPE
# ------------------------------------------------------------

print("Final Shape:")

print(
    df_analysis_ready.shape
)

print()


# ------------------------------------------------------------
# 16.2 FINAL INFORMATION
# ------------------------------------------------------------

print("Final Information:")

df_analysis_ready.info()

print()


# ------------------------------------------------------------
# 16.3 FINAL NULL VALUES
# ------------------------------------------------------------

print("Final NULL Values:")

print(
    df_analysis_ready.isnull().sum()
)

print()


# ------------------------------------------------------------
# 16.4 FINAL DUPLICATE ROWS
# ------------------------------------------------------------

print("Final Duplicate Rows:")

print(
    df_analysis_ready.duplicated().sum()
)

print()


# ------------------------------------------------------------
# 16.5 FINAL SALARY STATISTICS
# ------------------------------------------------------------

print("Final Salary Statistics:")

print(
    df_analysis_ready[
        "annual_salary"
    ].describe()
)

print()

In [ ]:
# ============================================================
# 17. EXPORT CLEAN DATASET
# ============================================================

print("=" * 60)
print("EXPORT CLEAN DATASET")
print("=" * 60)


# ------------------------------------------------------------
# 17.1 DEFINE OUTPUT FILE
# ------------------------------------------------------------

OUTPUT_FILE = "salary_survey_cleaned.csv"


# ------------------------------------------------------------
# 17.2 EXPORT DATASET
# ------------------------------------------------------------

df_analysis_ready.to_csv(
    OUTPUT_FILE,
    index=False
)


# ------------------------------------------------------------
# 17.3 CONFIRM EXPORT
# ------------------------------------------------------------

print("Export complete.")

print("\nClean dataset saved as:")

print(OUTPUT_FILE)


print("\nFinal dataset shape:")

print(
    df_analysis_ready.shape
)


print("\nData cleaning completed successfully.")

print("=" * 60)
```


SyntaxError: invalid syntax (3792655497.py, line 18)